In [1]:
import pandas as pd
import plotly.graph_objs as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output


# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set

filepath = 'Data/Console_share_'
df = []
for region in regions:
	filepath_curr = filepath + region + '.csv'
	df_temp = pd.read_csv(filepath_curr)
	rows = df_temp.shape[0]
	df_temp['Region'] = [regions[region] for _ in range(rows)]
	df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')


# Dash Set up
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

# Dropdown list and radio button options set up
region_dropdown = [{'label': regions[region], 'value': regions[region]} \
                   for region in regions]
vis_type_options = [{'label': 'Line Chart', 'value': 'line'},
                {'label': 'Bar Chart', 'value': 'bar'}]

# Declare headline and description
headline = 'Gaming Console Marketshare in 2018'
description = '''
                  The marketshare of gaming console is different in 
                  every corner in the world. For exmample, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph. 
              '''

# Dashboard layout
app.layout = html.Div([
	html.H1(children=headline, style={'text-align':'center'}), 
	# Position 0, headline
	html.Div(children=description, style={'width':'60%'}), 
	# Position 1, descritpion
	html.Div([
		html.Div([
			html.P('Select Region:'),
			dcc.Dropdown(
				id='region-dropdown',
				options=region_dropdown,
				value='World Wide',
				style={'width': '90%'}
				)
			], style={'width':'60%', 'display': 'inline-block',
			          'vertical-align': 'middle'}),
		html.Div([
			html.P('Select Visualization:'),
			dcc.RadioItems(
				id='type-radio',
				options=vis_type_options,
				value='line'
				)
			], style={'width':'40%', 'display': 'inline-block',
			          'vertical-align': 'middle'})
	]), # Position 2, Options
	html.Div(dcc.Graph(id='vis')), # Position 3, Graph
	html.Div(dcc.Markdown(id='markdown'), style={'width':'60%'}) 
	# Position 4, Markdown
]) # End Dashboard Div

def getLineChart(df, title):
	traces = []
	for console in df.columns:
		if console != 'Date' and console != 'Region':
			traces.append(dict(
				x=df['Date'],
				y=df[console],
				mode='lines+markers',
				marker={
						'size': 15,
						'line': {'width': 0.5, 'color': 'white'}
				},
				name=console
				))
	layout = dict(title=title, 
		          xaxis={'title':'Date'},
		          yaxis={'title':'Market Share (%)', 'range': [0,100]},
		          transition={'duration': 500})
	return {'data': traces, 'layout': layout}

def getBarChart(df, title):
	df = df.drop(['Region'], axis=1)
	data = []
	for console in df.columns:
		if console != 'Date':
			temp = {}
			temp['x'] = df['Date']
			temp['y'] = df[console]
			temp['type'] = 'bar'
			temp['name'] = console
			data.append(temp)

	layout = dict(title=title, 
		          xaxis={'title':'Consoles'},
		          yaxis={'title':'Market Share (%)', 'range': [0,100]},
		          barmode='group',
		          transition={'duration': 500})

	return {'data': data, 'layout': layout}

@app.callback([Output('vis','figure'), Output('markdown','children')],
	          [Input('region-dropdown','value'),
	           Input('type-radio','value')])
def display_graph(region, vis_type):
	df_temp = df[df['Region']==region]
	fig = None
	vis_title = 'Gaming Market Share in 2018'
	if vis_type == 'line':
		fig = getLineChart(df_temp, vis_title)
	elif vis_type == 'bar':
		fig = getBarChart(df_temp, vis_title)
	filepath_markdown = 'Data/Markdown_'
	filepath_markdown += region2brief[region]
	filepath_markdown += '.txt'
	f = open(filepath_markdown, 'r')
	text = 'A brief analysis: '
	text += f.read()
	f.close()
	return fig, text

if __name__ == '__main__':
    app.run_server(debug=True, host='0.0.0.0', port=1200)

C:\Users\Haz\AppData\Local\Temp\ipykernel_34804\1581337929.py:4: UserWarning: 
The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`
  import dash_core_components as dcc
C:\Users\Haz\AppData\Local\Temp\ipykernel_34804\1581337929.py:5: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html


ConnectionError: HTTPConnectionPool(host='0.0.0.0', port=1200): Max retries exceeded with url: /_alive_64e270db-7ea7-46b6-b513-69eddc3f32c4 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001A3F6E72120>: Failed to establish a new connection: [WinError 10049] The requested address is not valid in its context'))

In [1]:
import pandas as pd
import plotly.graph_objs as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import dash_auth
from datetime import datetime
from flask import request

# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set
filepath = 'Data/Console_share_'
df = []
for region in regions:
    filepath_curr = filepath + region + '.csv'
    df_temp = pd.read_csv(filepath_curr)
    rows = df_temp.shape[0]
    df_temp['Region'] = [regions[region] for _ in range(rows)]
    df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')

# Dash Set up
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

# Basic authentication setup
VALID_USERNAME_PASSWORD_PAIRS = {
    'harry': 'harry123'
}

auth = dash_auth.BasicAuth(
    app,
    VALID_USERNAME_PASSWORD_PAIRS
)

# Function to log login attempts
def log_login(username):
    with open('login_log.txt', 'a') as log_file:
        log_file.write(f"{datetime.now()} - {username} logged in\n")

# Middleware to log login attempts
@app.server.before_request
def before_request():
    auth_header = request.headers.get('Authorization')
    if auth_header:
        auth_type, auth_info = auth_header.split(None, 1)
        username, password = auth_info.strip().decode('base64').split(':', 1)
        if VALID_USERNAME_PASSWORD_PAIRS.get(username) == password:
            log_login(username)

# Dropdown list and radio button options set up
region_dropdown = [{'label': regions[region], 'value': regions[region]} \
                   for region in regions]
vis_type_options = [{'label': 'Line Chart', 'value': 'line'},
                {'label': 'Bar Chart', 'value': 'bar'}]

# Declare headline and description
headline = 'Gaming Console Marketshare in 2018'
description = '''
                  The marketshare of gaming console is different in 
                  every corner in the world. For exmample, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph. 
              '''

# Dashboard layout
app.layout = html.Div([
    html.H1(children=headline, style={'text-align':'center'}), 
    # Position 0, headline
    html.Div(children=description, style={'width':'60%'}), 
    # Position 1, descritpion
    html.Div([
        html.Div([
            html.P('Select Region:'),
            dcc.Dropdown(
                id='region-dropdown',
                options=region_dropdown,
                value='World Wide',
                style={'width': '90%'}
                )
            ], style={'width':'60%', 'display': 'inline-block',
                      'vertical-align': 'middle'}),
        html.Div([
            html.P('Select Visualization:'),
            dcc.RadioItems(
                id='type-radio',
                options=vis_type_options,
                value='line'
                )
            ], style={'width':'40%', 'display': 'inline-block',
                      'vertical-align': 'middle'})
    ]), # Position 2, Options
    html.Div(dcc.Graph(id='vis')), # Position 3, Graph
    html.Div(dcc.Markdown(id='markdown'), style={'width':'60%'}) 
    # Position 4, Markdown
]) # End Dashboard Div

def getLineChart(df, title):
    traces = []
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            traces.append(dict(
                x=df['Date'],
                y=df[console],
                mode='lines+markers',
                marker={
                        'size': 15,
                        'line': {'width': 0.5, 'color': 'white'}
                },
                name=console
                ))
    layout = dict(title=title, 
                  xaxis={'title':'Date'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  transition={'duration': 500})
    return {'data': traces, 'layout': layout}

def getBarChart(df, title):
    df = df.drop(['Region'], axis=1)
    data = []
    for console in df.columns:
        if console != 'Date':
            temp = {}
            temp['x'] = df['Date']
            temp['y'] = df[console]
            temp['type'] = 'bar'
            temp['name'] = console
            data.append(temp)

    layout = dict(title=title, 
                  xaxis={'title':'Consoles'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  barmode='group',
                  transition={'duration': 500})

    return {'data': data, 'layout': layout}

@app.callback([Output('vis','figure'), Output('markdown','children')],
              [Input('region-dropdown','value'),
               Input('type-radio','value')])
def display_graph(region, vis_type):
    df_temp = df[df['Region']==region]
    fig = None
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'line':
        fig = getLineChart(df_temp, vis_title)
    elif vis_type == 'bar':
        fig = getBarChart(df_temp, vis_title)
    filepath_markdown = 'Data/Markdown_'
    filepath_markdown += region2brief[region]
    filepath_markdown += '.txt'
    f = open(filepath_markdown, 'r')
    text = 'A brief analysis: '
    text += f.read()
    f.close()
    return fig, text

if __name__ == '__main__':
    app.run_server(debug=True, port=1200)

C:\Users\Haz\AppData\Local\Temp\ipykernel_31532\3284396347.py:4: UserWarning: 
The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`
  import dash_core_components as dcc
C:\Users\Haz\AppData\Local\Temp\ipykernel_31532\3284396347.py:5: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



...

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
AttributeError: 'str' object has no attribute 'decode'



In [ ]:
import pandas as pd
import plotly.graph_objs as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import dash_auth
from datetime import datetime
from flask import request
import base64

# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set
filepath = 'Data/Console_share_'
df = []
for region in regions:
    filepath_curr = filepath + region + '.csv'
    df_temp = pd.read_csv(filepath_curr)
    rows = df_temp.shape[0]
    df_temp['Region'] = [regions[region] for _ in range(rows)]
    df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')

# Dash Set up
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

# Basic authentication setup
VALID_USERNAME_PASSWORD_PAIRS = {
    'harry': 'harry123'
}

auth = dash_auth.BasicAuth(
    app,
    VALID_USERNAME_PASSWORD_PAIRS
)

# Function to log login attempts
def log_login(username):
    with open('login_log.txt', 'a') as log_file:
        log_file.write(f"{datetime.now()} - {username} logged in\n")

# Middleware to log login attempts
@app.server.before_request
def before_request():
    auth_header = request.headers.get('Authorization')
    if auth_header:
        auth_type, auth_info = auth_header.split(None, 1)
        username, password = base64.b64decode(auth_info).decode('utf-8').split(':', 1)
        if VALID_USERNAME_PASSWORD_PAIRS.get(username) == password:
            log_login(username)

# Dropdown list and radio button options set up
region_dropdown = [{'label': regions[region], 'value': regions[region]} \
                   for region in regions]
vis_type_options = [{'label': 'Line Chart', 'value': 'line'},
                {'label': 'Bar Chart', 'value': 'bar'}]

# Declare headline and description
headline = 'Gaming Console Marketshare in 2018'
description = '''
                  The marketshare of gaming console is different in 
                  every corner in the world. For exmample, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph. 
              '''

# Dashboard layout
app.layout = html.Div([
    html.H1(children=headline, style={'text-align':'center'}), 
    # Position 0, headline
    html.Div(children=description, style={'width':'60%'}), 
    # Position 1, descritpion
    html.Div([
        html.Div([
            html.P('Select Region:'),
            dcc.Dropdown(
                id='region-dropdown',
                options=region_dropdown,
                value='World Wide',
                style={'width': '90%'}
                )
            ], style={'width':'60%', 'display': 'inline-block',
                      'vertical-align': 'middle'}),
        html.Div([
            html.P('Select Visualization:'),
            dcc.RadioItems(
                id='type-radio',
                options=vis_type_options,
                value='line'
                )
            ], style={'width':'40%', 'display': 'inline-block',
                      'vertical-align': 'middle'})
    ]), # Position 2, Options
    html.Div(dcc.Graph(id='vis')), # Position 3, Graph
    html.Div(dcc.Markdown(id='markdown'), style={'width':'60%'}) 
    # Position 4, Markdown
]) # End Dashboard Div

def getLineChart(df, title):
    traces = []
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            traces.append(dict(
                x=df['Date'],
                y=df[console],
                mode='lines+markers',
                marker={
                        'size': 15,
                        'line': {'width': 0.5, 'color': 'white'}
                },
                name=console
                ))
    layout = dict(title=title, 
                  xaxis={'title':'Date'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  transition={'duration': 500})
    return {'data': traces, 'layout': layout}

def getBarChart(df, title):
    df = df.drop(['Region'], axis=1)
    data = []
    for console in df.columns:
        if console != 'Date':
            temp = {}
            temp['x'] = df['Date']
            temp['y'] = df[console]
            temp['type'] = 'bar'
            temp['name'] = console
            data.append(temp)

    layout = dict(title=title, 
                  xaxis={'title':'Consoles'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  barmode='group',
                  transition={'duration': 500})

    return {'data': data, 'layout': layout}

@app.callback([Output('vis','figure'), Output('markdown','children')],
              [Input('region-dropdown','value'),
               Input('type-radio','value')])
def display_graph(region, vis_type):
    df_temp = df[df['Region']==region]
    fig = None
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'line':
        fig = getLineChart(df_temp, vis_title)
    elif vis_type == 'bar':
        fig = getBarChart(df_temp, vis_title)
    filepath_markdown = 'Data/Markdown_'
    filepath_markdown += region2brief[region]
    filepath_markdown += '.txt'
    with open(filepath_markdown, 'r') as f:
        text = 'A brief analysis: '
        text += f.read()
    return fig, text

if __name__ == '__main__':
    app.run_server(debug=True, port=1200)

Exception: Login Required

In [4]:
import pandas as pd
import plotly.graph_objs as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import dash_auth
from datetime import datetime
from flask import request, session
import base64
import os

# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set
filepath = 'Data/Console_share_'
df = []
for region in regions:
    filepath_curr = filepath + region + '.csv'
    df_temp = pd.read_csv(filepath_curr)
    rows = df_temp.shape[0]
    df_temp['Region'] = [regions[region] for _ in range(rows)]
    df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')

# Dash Set up
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
server = app.server
server.secret_key = os.urandom(24)  # Needed for session management

# Basic authentication setup
VALID_USERNAME_PASSWORD_PAIRS = {
    'harry': 'harry123'
}

auth = dash_auth.BasicAuth(
    app,
    VALID_USERNAME_PASSWORD_PAIRS
)

# Function to log login attempts
def log_login(username):
    with open('login_log.txt', 'a') as log_file:
        log_file.write(f"{datetime.now()} - {username} logged in\n")

# Middleware to log login attempts
@app.server.before_request
def before_request():
    auth_header = request.headers.get('Authorization')
    if auth_header:
        auth_type, auth_info = auth_header.split(None, 1)
        username, password = base64.b64decode(auth_info).decode('utf-8').split(':', 1)
        if VALID_USERNAME_PASSWORD_PAIRS.get(username) == password:
            if not session.get('logged_in'):
                log_login(username)
                session['logged_in'] = True
            else:
                session['logged_in'] = False

# Dropdown list and radio button options set up
region_dropdown = [{'label': regions[region], 'value': regions[region]} \
                   for region in regions]
vis_type_options = [{'label': 'Line Chart', 'value': 'line'},
                {'label': 'Bar Chart', 'value': 'bar'}]

# Declare headline and description
headline = 'Gaming Console Marketshare in 2018'
description = '''
                  The marketshare of gaming console is different in 
                  every corner in the world. For exmample, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph. 
              '''

# Dashboard layout
app.layout = html.Div([
    html.H1(children=headline, style={'text-align':'center'}), 
    # Position 0, headline
    html.Div(children=description, style={'width':'60%'}), 
    # Position 1, descritpion
    html.Div([
        html.Div([
            html.P('Select Region:'),
            dcc.Dropdown(
                id='region-dropdown',
                options=region_dropdown,
                value='World Wide',
                style={'width': '90%'}
                )
            ], style={'width':'60%', 'display': 'inline-block',
                      'vertical-align': 'middle'}),
        html.Div([
            html.P('Select Visualization:'),
            dcc.RadioItems(
                id='type-radio',
                options=vis_type_options,
                value='line'
                )
            ], style={'width':'40%', 'display': 'inline-block',
                      'vertical-align': 'middle'})
    ]), # Position 2, Options
    html.Div(dcc.Graph(id='vis')), # Position 3, Graph
    html.Div(dcc.Markdown(id='markdown'), style={'width':'60%'}) 
    # Position 4, Markdown
]) # End Dashboard Div

def getLineChart(df, title):
    traces = []
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            traces.append(dict(
                x=df['Date'],
                y=df[console],
                mode='lines+markers',
                marker={
                        'size': 15,
                        'line': {'width': 0.5, 'color': 'white'}
                },
                name=console
                ))
    layout = dict(title=title, 
                  xaxis={'title':'Date'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  transition={'duration': 500})
    return {'data': traces, 'layout': layout}

def getBarChart(df, title):
    df = df.drop(['Region'], axis=1)
    data = []
    for console in df.columns:
        if console != 'Date':
            temp = {}
            temp['x'] = df['Date']
            temp['y'] = df[console]
            temp['type'] = 'bar'
            temp['name'] = console
            data.append(temp)

    layout = dict(title=title, 
                  xaxis={'title':'Consoles'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  barmode='group',
                  transition={'duration': 500})

    return {'data': data, 'layout': layout}

@app.callback([Output('vis','figure'), Output('markdown','children')],
              [Input('region-dropdown','value'),
               Input('type-radio','value')])
def display_graph(region, vis_type):
    df_temp = df[df['Region']==region]
    fig = None
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'line':
        fig = getLineChart(df_temp, vis_title)
    elif vis_type == 'bar':
        fig = getBarChart(df_temp, vis_title)
    filepath_markdown = 'Data/Markdown_'
    filepath_markdown += region2brief[region]
    filepath_markdown += '.txt'
    with open(filepath_markdown, 'r') as f:
        text = 'A brief analysis: '
        text += f.read()
    return fig, text

if __name__ == '__main__':
    app.run_server(debug=True, port=1200)

Exception: Login Required

In [5]:
import pandas as pd
import plotly.graph_objs as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import dash_auth
from datetime import datetime
from flask import request, session
import base64
import os

# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set
filepath = 'Data/Console_share_'
df = []
for region in regions:
    filepath_curr = filepath + region + '.csv'
    df_temp = pd.read_csv(filepath_curr)
    rows = df_temp.shape[0]
    df_temp['Region'] = [regions[region] for _ in range(rows)]
    df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')

# Dash Set up
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
server = app.server
server.secret_key = os.urandom(24)  # Needed for session management

# Basic authentication setup
VALID_USERNAME_PASSWORD_PAIRS = {
    'harry': 'harry123'
}

auth = dash_auth.BasicAuth(
    app,
    VALID_USERNAME_PASSWORD_PAIRS
)

# Function to log login attempts
def log_login(username):
    with open('login_log.txt', 'a') as log_file:
        log_file.write(f"{datetime.now()} - {username} logged in\n")

# Middleware to log login attempts
@app.server.before_request
def before_request():
    auth_header = request.headers.get('Authorization')
    if auth_header:
        auth_type, auth_info = auth_header.split(None, 1)
        username, password = base64.b64decode(auth_info).decode('utf-8').split(':', 1)
        if VALID_USERNAME_PASSWORD_PAIRS.get(username) == password:
            if not session.get('logged_in'):
                log_login(username)
                session['logged_in'] = True
            elif session.get('username') != username:
                log_login(username)
                session['username'] = username

# Dropdown list and radio button options set up
region_dropdown = [{'label': regions[region], 'value': regions[region]} \
                   for region in regions]
vis_type_options = [{'label': 'Line Chart', 'value': 'line'},
                {'label': 'Bar Chart', 'value': 'bar'}]

# Declare headline and description
headline = 'Gaming Console Marketshare in 2018'
description = '''
                  The marketshare of gaming console is different in 
                  every corner in the world. For exmample, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph. 
              '''

# Dashboard layout
app.layout = html.Div([
    html.H1(children=headline, style={'text-align':'center'}), 
    # Position 0, headline
    html.Div(children=description, style={'width':'60%'}), 
    # Position 1, descritpion
    html.Div([
        html.Div([
            html.P('Select Region:'),
            dcc.Dropdown(
                id='region-dropdown',
                options=region_dropdown,
                value='World Wide',
                style={'width': '90%'}
                )
            ], style={'width':'60%', 'display': 'inline-block',
                      'vertical-align': 'middle'}),
        html.Div([
            html.P('Select Visualization:'),
            dcc.RadioItems(
                id='type-radio',
                options=vis_type_options,
                value='line'
                )
            ], style={'width':'40%', 'display': 'inline-block',
                      'vertical-align': 'middle'})
    ]), # Position 2, Options
    html.Div(dcc.Graph(id='vis')), # Position 3, Graph
    html.Div(dcc.Markdown(id='markdown'), style={'width':'60%'}) 
    # Position 4, Markdown
]) # End Dashboard Div

def getLineChart(df, title):
    traces = []
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            traces.append(dict(
                x=df['Date'],
                y=df[console],
                mode='lines+markers',
                marker={
                        'size': 15,
                        'line': {'width': 0.5, 'color': 'white'}
                },
                name=console
                ))
    layout = dict(title=title, 
                  xaxis={'title':'Date'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  transition={'duration': 500})
    return {'data': traces, 'layout': layout}

def getBarChart(df, title):
    df = df.drop(['Region'], axis=1)
    data = []
    for console in df.columns:
        if console != 'Date':
            temp = {}
            temp['x'] = df['Date']
            temp['y'] = df[console]
            temp['type'] = 'bar'
            temp['name'] = console
            data.append(temp)

    layout = dict(title=title, 
                  xaxis={'title':'Consoles'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  barmode='group',
                  transition={'duration': 500})

    return {'data': data, 'layout': layout}

@app.callback([Output('vis','figure'), Output('markdown','children')],
              [Input('region-dropdown','value'),
               Input('type-radio','value')])
def display_graph(region, vis_type):
    df_temp = df[df['Region']==region]
    fig = None
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'line':
        fig = getLineChart(df_temp, vis_title)
    elif vis_type == 'bar':
        fig = getBarChart(df_temp, vis_title)
    filepath_markdown = 'Data/Markdown_'
    filepath_markdown += region2brief[region]
    filepath_markdown += '.txt'
    with open(filepath_markdown, 'r') as f:
        text = 'A brief analysis: '
        text += f.read()
    return fig, text

if __name__ == '__main__':
    app.run_server(debug=True, port=1200)

Exception: Login Required

### oauth

In [4]:
import pandas as pd
import plotly.graph_objs as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
from datetime import datetime
from flask import Flask, redirect, url_for, session
from authlib.integrations.flask_client import OAuth
import os

# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set
filepath = 'Data/Console_share_'
df = []
for region in regions:
    filepath_curr = filepath + region + '.csv'
    df_temp = pd.read_csv(filepath_curr)
    rows = df_temp.shape[0]
    df_temp['Region'] = [regions[region] for _ in range(rows)]
    df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')

# Flask server setup
server = Flask(__name__)
server.secret_key = os.urandom(24)

# OAuth setup
oauth = OAuth(server)
google = oauth.register(
    name='google',
    client_id='YOUR_GOOGLE_CLIENT_ID',
    client_secret='YOUR_GOOGLE_CLIENT_SECRET',
    access_token_url='https://accounts.google.com/o/oauth2/token',
    access_token_params=None,
    authorize_url='https://accounts.google.com/o/oauth2/auth',
    authorize_params=None,
    api_base_url='https://www.googleapis.com/oauth2/v1/',
    userinfo_endpoint='https://www.googleapis.com/oauth2/v1/userinfo',
    client_kwargs={'scope': 'openid profile email'},
)

# Dash setup
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, server=server, external_stylesheets=external_stylesheets)

# Function to log login attempts
def log_login(username):
    with open('login_log.txt', 'a') as log_file:
        log_file.write(f"{datetime.now()} - {username} logged in\n")

# Middleware to log login attempts
@app.server.before_request
def before_request():
    if 'google_token' in session:
        user_info = google.get('userinfo').json()
        username = user_info['email']
        if not session.get('logged_in'):
            log_login(username)
            session['logged_in'] = True
        elif session.get('username') != username:
            log_login(username)
            session['username'] = username
    else:
        return redirect(url_for('login'))

# OAuth login route
@app.server.route('/login')
def login():
    redirect_uri = url_for('authorize', _external=True)
    return google.authorize_redirect(redirect_uri)

# OAuth authorize route
@app.server.route('/authorize')
def authorize():
    token = google.authorize_access_token()
    session['google_token'] = token
    user_info = google.get('userinfo').json()
    session['username'] = user_info['email']
    return redirect('/')

# Dropdown list and radio button options set up
region_dropdown = [{'label': regions[region], 'value': regions[region]} \
                   for region in regions]
vis_type_options = [{'label': 'Line Chart', 'value': 'line'},
                {'label': 'Bar Chart', 'value': 'bar'}]

# Declare headline and description
headline = 'Gaming Console Marketshare in 2018'
description = '''
                  The marketshare of gaming console is different in 
                  every corner in the world. For exmample, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph. 
              '''

# Dashboard layout
app.layout = html.Div([
    html.H1(children=headline, style={'text-align':'center'}), 
    # Position 0, headline
    html.Div(children=description, style={'width':'60%'}), 
    # Position 1, descritpion
    html.Div([
        html.Div([
            html.P('Select Region:'),
            dcc.Dropdown(
                id='region-dropdown',
                options=region_dropdown,
                value='World Wide',
                style={'width': '90%'}
                )
            ], style={'width':'60%', 'display': 'inline-block',
                      'vertical-align': 'middle'}),
        html.Div([
            html.P('Select Visualization:'),
            dcc.RadioItems(
                id='type-radio',
                options=vis_type_options,
                value='line'
                )
            ], style={'width':'40%', 'display': 'inline-block',
                      'vertical-align': 'middle'})
    ]), # Position 2, Options
    html.Div(dcc.Graph(id='vis')), # Position 3, Graph
    html.Div(dcc.Markdown(id='markdown'), style={'width':'60%'}) 
    # Position 4, Markdown
]) # End Dashboard Div

def getLineChart(df, title):
    traces = []
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            traces.append(dict(
                x=df['Date'],
                y=df[console],
                mode='lines+markers',
                marker={
                        'size': 15,
                        'line': {'width': 0.5, 'color': 'white'}
                },
                name=console
                ))
    layout = dict(title=title, 
                  xaxis={'title':'Date'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  transition={'duration': 500})
    return {'data': traces, 'layout': layout}

def getBarChart(df, title):
    df = df.drop(['Region'], axis=1)
    data = []
    for console in df.columns:
        if console != 'Date':
            temp = {}
            temp['x'] = df['Date']
            temp['y'] = df[console]
            temp['type'] = 'bar'
            temp['name'] = console
            data.append(temp)

    layout = dict(title=title, 
                  xaxis={'title':'Consoles'},
                  yaxis={'title':'Market Share (%)', 'range': [0,100]},
                  barmode='group',
                  transition={'duration': 500})

    return {'data': data, 'layout': layout}

@app.callback([Output('vis','figure'), Output('markdown','children')],
              [Input('region-dropdown','value'),
               Input('type-radio','value')])
def display_graph(region, vis_type):
    df_temp = df[df['Region']==region]
    fig = None
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'line':
        fig = getLineChart(df_temp, vis_title)
    elif vis_type == 'bar':
        fig = getBarChart(df_temp, vis_title)
    filepath_markdown = 'Data/Markdown_'
    filepath_markdown += region2brief[region]
    filepath_markdown += '.txt'
    with open(filepath_markdown, 'r') as f:
        text = 'A brief analysis: '
        text += f.read()
    return fig, text

if __name__ == '__main__':
    app.run_server(debug=True, port=1200)

TooManyRedirects: Exceeded 30 redirects.

In [6]:
import pandas as pd
from bokeh.plotting import figure, output_file, save
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, Select, RadioButtonGroup, Div
from bokeh.io import curdoc
from bokeh.embed import file_html
from bokeh.resources import CDN
from flask import Flask, redirect, url_for, session
from authlib.integrations.flask_client import OAuth
import os

# Define the regions and the full name
regions = {'CA': 'Canada', 'GB': 'Great Britain', 'HK': 'Hong Kong',
           'JP': 'Japan', 'KR': 'South Korea', 'US': 'United States',
           'WorldWide': 'World Wide'}
region2brief = dict([(regions[k], k) for k in regions])

# Read multiple data set
filepath = 'Data/Console_share_'
df = []
for region in regions:
    filepath_curr = filepath + region + '.csv'
    df_temp = pd.read_csv(filepath_curr)
    rows = df_temp.shape[0]
    df_temp['Region'] = [regions[region] for _ in range(rows)]
    df.append(df_temp)

df = pd.concat(df)
df = df.drop('Other', axis=1)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')

# Flask server setup
server = Flask(__name__)
server.secret_key = os.urandom(24)

# OAuth setup
oauth = OAuth(server)
google = oauth.register(
    name='google',
    client_id='YOUR_GOOGLE_CLIENT_ID',
    client_secret='YOUR_GOOGLE_CLIENT_SECRET',
    access_token_url='https://accounts.google.com/o/oauth2/token',
    access_token_params=None,
    authorize_url='https://accounts.google.com/o/oauth2/auth',
    authorize_params=None,
    api_base_url='https://www.googleapis.com/oauth2/v1/',
    userinfo_endpoint='https://www.googleapis.com/oauth2/v1/userinfo',
    client_kwargs={'scope': 'openid profile email'},
)

# Function to log login attempts
def log_login(username):
    with open('login_log.txt', 'a') as log_file:
        log_file.write(f"{datetime.now()} - {username} logged in\n")

# Middleware to log login attempts
@server.before_request
def before_request():
    if 'google_token' in session:
        user_info = google.get('userinfo').json()
        username = user_info['email']
        if not session.get('logged_in'):
            log_login(username)
            session['logged_in'] = True
        elif session.get('username') != username:
            log_login(username)
            session['username'] = username
    else:
        return redirect(url_for('login'))

# OAuth login route
@server.route('/login')
def login():
    redirect_uri = url_for('authorize', _external=True)
    return google.authorize_redirect(redirect_uri)

# OAuth authorize route
@server.route('/authorize')
def authorize():
    token = google.authorize_access_token()
    session['google_token'] = token
    user_info = google.get('userinfo').json()
    session['username'] = user_info['email']
    return redirect('/')

# Bokeh setup
output_file("dashboard.html")

# Dropdown list and radio button options set up
region_dropdown = Select(title="Select Region:", value="World Wide", options=list(regions.values()))
vis_type_options = RadioButtonGroup(labels=["Line Chart", "Bar Chart"], active=0)

# Declare headline and description
headline = Div(text="<h1>Gaming Console Marketshare in 2018</h1>", style={'text-align': 'center'})
description = Div(text="""
                  <p>The marketshare of gaming console is different in 
                  every corner in the world. For example, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph.</p>
              """, style={'width': '60%'})

# Function to create line chart
def get_line_chart(df, title):
    p = figure(title=title, x_axis_type='datetime', plot_height=400, plot_width=800)
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            p.line(df['Date'], df[console], legend_label=console, line_width=2)
    p.xaxis.axis_label = 'Date'
    p.yaxis.axis_label = 'Market Share (%)'
    p.legend.location = "top_left"
    return p

# Function to create bar chart
def get_bar_chart(df, title):
    p = figure(title=title, x_axis_type='datetime', plot_height=400, plot_width=800)
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            p.vbar(x=df['Date'], top=df[console], width=0.9, legend_label=console)
    p.xaxis.axis_label = 'Date'
    p.yaxis.axis_label = 'Market Share (%)'
    p.legend.location = "top_left"
    return p

# Callback function to update the graph
def update(attr, old, new):
    region = region_dropdown.value
    vis_type = vis_type_options.labels[vis_type_options.active]
    df_temp = df[df['Region'] == region]
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'Line Chart':
        plot = get_line_chart(df_temp, vis_title)
    else:
        plot = get_bar_chart(df_temp, vis_title)
    layout.children[3] = plot

region_dropdown.on_change('value', update)
vis_type_options.on_change('active', update)

# Initial plot
initial_plot = get_line_chart(df[df['Region'] == 'World Wide'], 'Gaming Market Share in 2018')

# Layout
layout = column(headline, description, region_dropdown, vis_type_options, initial_plot)

# Add layout to current document
curdoc().add_root(layout)

# Save the dashboard as a static HTML file
save(layout)

# Generate HTML content
html_content = file_html(layout, CDN, "Gaming Console Marketshare Dashboard")

# Save HTML content to a file
with open("dashboard.html", "w") as f:
    f.write(html_content)

AttributeError: unexpected attribute 'style' to Div, similar attributes are styles, stylesheets or syncable

In [11]:
import pandas as pd
from bokeh.plotting import figure, output_file, save
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, Select, RadioButtonGroup, Div
from bokeh.io import curdoc
from bokeh.embed import file_html
from bokeh.resources import CDN

# Define dummy data
regions = ['World Wide', 'US', 'JP', 'EU']
data = {
    'Date': pd.date_range(start='1/1/2018', periods=12, freq='ME'),
    'Console1': [20, 30, 40, 50, 60, 70, 80, 70, 60, 50, 40, 30],
    'Console2': [80, 70, 60, 50, 40, 30, 20, 30, 40, 50, 60, 70],
    'Region': ['World Wide'] * 12
}
df = pd.DataFrame(data)

# Add dummy data for other regions
for region in regions[1:]:
    temp_data = data.copy()
    temp_data['Region'] = [region] * 12
    temp_df = pd.DataFrame(temp_data)
    df = pd.concat([df, temp_df])

# Bokeh setup
output_file("dashboard.html")

# Dropdown list and radio button options set up
region_dropdown = Select(title="Select Region:", value="World Wide", options=regions)
vis_type_options = RadioButtonGroup(labels=["Line Chart", "Bar Chart"], active=0)

# Declare headline and description
headline = Div(text="<h1>Gaming Console Marketshare in 2018</h1>", style={'text-align': 'center'})
description = Div(text="""
                  <p>The marketshare of gaming console is different in 
                  every corner in the world. For example, the marketshare
                  of Xbox in Japan is significantly lower than the marketshare
                  in the US. In this dashboard, you may select the region in 
                  the dropdown list below and the visualization. A brief 
                  analysis of the gaming market share in the 
                  selected region will be displayed under the graph.</p>
              """, style={'width': '60%'})

# Function to create line chart
def get_line_chart(df, title):
    p = figure(title=title, x_axis_type='datetime', plot_height=400, plot_width=800)
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            p.line(df['Date'], df[console], legend_label=console, line_width=2)
    p.xaxis.axis_label = 'Date'
    p.yaxis.axis_label = 'Market Share (%)'
    p.legend.location = "top_left"
    return p

# Function to create bar chart
def get_bar_chart(df, title):
    p = figure(title=title, x_axis_type='datetime', plot_height=400, plot_width=800)
    for console in df.columns:
        if console != 'Date' and console != 'Region':
            p.vbar(x=df['Date'], top=df[console], width=0.9, legend_label=console)
    p.xaxis.axis_label = 'Date'
    p.yaxis.axis_label = 'Market Share (%)'
    p.legend.location = "top_left"
    return p

# Callback function to update the graph
def update(attr, old, new):
    region = region_dropdown.value
    vis_type = vis_type_options.labels[vis_type_options.active]
    df_temp = df[df['Region'] == region]
    vis_title = 'Gaming Market Share in 2018'
    if vis_type == 'Line Chart':
        plot = get_line_chart(df_temp, vis_title)
    else:
        plot = get_bar_chart(df_temp, vis_title)
    layout.children[4] = plot

region_dropdown.on_change('value', update)
vis_type_options.on_change('active', update)

# Initial plot
initial_plot = get_line_chart(df[df['Region'] == 'World Wide'], 'Gaming Market Share in 2018')

# Layout
layout = column(headline, description, region_dropdown, vis_type_options, initial_plot)

# Add layout to current document
curdoc().add_root(layout)

# Save the dashboard as a static HTML file
save(layout)

# Generate HTML content
html_content = file_html(layout, CDN, "Gaming Console Marketshare Dashboard")

# Save HTML content to a file
with open("dashboard.html", "w") as f:
    f.write(html_content)

AttributeError: unexpected attribute 'style' to Div, similar attributes are styles, stylesheets or syncable